In [1]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display
from pathlib import Path

# -----------------------------
# User parameters
# -----------------------------
#nc_path = r"Y:\soc\l0c\2024\05\awe_l0c_q20_2024123T0728_02515_v01.nc"
nc_path = r"/Users/anhphan/Downloads/awe_l0c_q20_2024123T0728_02515_v01.nc"

out_dir = Path("Animation")
out_dir.mkdir(parents=True, exist_ok=True)

# North America map extent
LON_MIN = -180
LON_MAX = -90
LAT_MIN = -30
LAT_MAX = 55

# Display every N frames
FRAME_STEP = 50

VMIN = 3
VMAX = 14
SCATTER_SIZE = 2.0

# -----------------------------
# Load dataset
# -----------------------------
ds = Dataset(nc_path, "r")

rad_all = ds.variables["Radiance"][:]
lat_all = ds.variables["Latitude"][:]
lon_raw = ds.variables["Longitude"][:]

lon_all = ((lon_raw + 180) % 360) - 180

nt = rad_all.shape[0]
print(f"Radiance shape: {rad_all.shape}")
print(f"Number of frames: {nt}")

frames = list(range(0, nt, FRAME_STEP))
print(f"Animating every {FRAME_STEP} frames")
print(f"Number of animation frames: {len(frames)}")

# -----------------------------
# Precompute data inside North America box
# -----------------------------
lon_list = []
lat_list = []
rad_list = []

for t in frames:
    lat_t = lat_all[t, :, :]
    lon_t = lon_all[t, :, :]
    rad_t = rad_all[t, :, :]

    lat_flat = lat_t.ravel()
    lon_flat = lon_t.ravel()
    rad_flat = rad_t.ravel()

    valid = (
        np.isfinite(lat_flat)
        & np.isfinite(lon_flat)
        & np.isfinite(rad_flat)
    )

    inside_box = (
        (lat_flat >= LAT_MIN) & (lat_flat <= LAT_MAX)
        & (lon_flat >= LON_MIN) & (lon_flat <= LON_MAX)
    )

    mask = valid & inside_box

    lon_list.append(lon_flat[mask])
    lat_list.append(lat_flat[mask])
    rad_list.append(rad_flat[mask])

# -----------------------------
# Set up figure
# -----------------------------
proj = ccrs.PlateCarree()

fig = plt.figure(figsize=(12, 12))
ax = plt.axes(projection=proj)

ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())

ax.coastlines(linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)

gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# Initial scatter
scat = ax.scatter(
    lon_list[0],
    lat_list[0],
    c=rad_list[0],
    s=SCATTER_SIZE,
    cmap="gray",
    vmin=VMIN,
    vmax=VMAX,
    transform=ccrs.PlateCarree(),
)

# Smaller/thinner colorbar
cbar = plt.colorbar(
    scat,
    ax=ax,
    label="Radiance",
    fraction=0.02,
    pad=0.01,
    shrink=0.8,
    aspect=40,
)

title = ax.set_title(
    f"AWE Radiance over North America – Frame {frames[0]}"
)

plt.subplots_adjust(
    left=0.04,
    right=0.92,
    bottom=0.05,
    top=0.92,
)

# -----------------------------
# Animation function
# -----------------------------
def update(i):
    lon_v = lon_list[i]
    lat_v = lat_list[i]
    rad_v = rad_list[i]

    if lon_v.size == 0:
        scat.set_offsets(np.empty((0, 2)))
        scat.set_array(np.array([]))
    else:
        scat.set_offsets(np.column_stack((lon_v, lat_v)))
        scat.set_array(rad_v)

    frame_number = frames[i]
    title.set_text(
        f"AWE Radiance over North America – Frame {frame_number}"
    )

    return scat, title

anim = FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=250,
    blit=False,
)

# -----------------------------
# Show inline
# -----------------------------
plt.close(fig)
display(HTML(anim.to_jshtml()))

# -----------------------------
# Save as GIF
# -----------------------------
gif_filename = out_dir / "awe_radiance_north_america_every_25_frames.gif"

writer = PillowWriter(fps=3)
anim.save(gif_filename, writer=writer)

print(f"Saved GIF to {gif_filename}")

ds.close()

Radiance shape: (1756, 256, 256)
Number of frames: 1756
Animating every 50 frames
Number of animation frames: 36


/Users/anhphan/opt/anaconda3/envs/juwavelet_env/lib/python3.10/site-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_1_states_provinces_lakes.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/Users/anhphan/opt/anaconda3/envs/juwavelet_env/lib/python3.10/site-packages/shapely/creation.py:730: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/Users/anhphan/opt/anaconda3/envs/juwavelet_env/lib/python3.10/site-packages/shapely/creation.py:730: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/Users/anhphan/opt/anaconda3/envs/juwavelet_env/lib/python3.10/site-packages/shapely/creation.py:730: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, *

/Users/anhphan/opt/anaconda3/envs/juwavelet_env/lib/python3.10/site-packages/shapely/creation.py:730: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/Users/anhphan/opt/anaconda3/envs/juwavelet_env/lib/python3.10/site-packages/shapely/creation.py:730: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/Users/anhphan/opt/anaconda3/envs/juwavelet_env/lib/python3.10/site-packages/shapely/creation.py:730: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/Users/anhphan/opt/anaconda3/envs/juwavelet_env/lib/python3.10/site-packages/shapely/creation.py:730: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/Users/anhphan/opt/anaconda3/envs/ju

Saved GIF to Animation/awe_radiance_north_america_every_25_frames.gif


In [17]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML, display
from pathlib import Path

# -----------------------------
# User parameters
# -----------------------------
nc_path = r"Y:\soc\l0c\2024\05\awe_l0c_q20_2024123T0728_02515_v01.nc"

out_dir = Path("Animation")
out_dir.mkdir(parents=True, exist_ok=True)

# Pacific-centered map extent
# 150E to 270E = 90W
LON_MIN = 165
LON_MAX = 270
LAT_MIN = -45
LAT_MAX = 55

FRAME_STEP = 25

VMIN = 3
VMAX = 20
SCATTER_SIZE = 2.0
FPS = 3

# -----------------------------
# Load dataset
# -----------------------------
ds = Dataset(nc_path, "r")

rad_all = ds.variables["Radiance"][:]
lat_all = ds.variables["Latitude"][:]
lon_raw = ds.variables["Longitude"][:]

# Keep longitude in 0–360 format
lon_all = lon_raw.copy()

# If longitude is accidentally stored as -180 to 180, convert to 0–360
lon_all = lon_all % 360

nt = rad_all.shape[0]
print(f"Radiance shape: {rad_all.shape}")
print(f"Number of frames: {nt}")

frames = list(range(0, nt, FRAME_STEP))
print(f"Animating every {FRAME_STEP} frames")
print(f"Number of animation frames: {len(frames)}")

# -----------------------------
# Precompute data inside map box
# -----------------------------
lon_list = []
lat_list = []
rad_list = []

for t in frames:
    lat_t = lat_all[t, :, :]
    lon_t = lon_all[t, :, :]
    rad_t = rad_all[t, :, :]

    lat_flat = lat_t.ravel()
    lon_flat = lon_t.ravel()
    rad_flat = rad_t.ravel()

    valid = (
        np.isfinite(lat_flat)
        & np.isfinite(lon_flat)
        & np.isfinite(rad_flat)
    )

    inside_box = (
        (lat_flat >= LAT_MIN) & (lat_flat <= LAT_MAX)
        & (lon_flat >= LON_MIN) & (lon_flat <= LON_MAX)
    )

    mask = valid & inside_box

    lon_list.append(lon_flat[mask])
    lat_list.append(lat_flat[mask])
    rad_list.append(rad_flat[mask])

# -----------------------------
# Set up figure
# -----------------------------
proj = ccrs.PlateCarree(central_longitude=180)

fig = plt.figure(figsize=(12, 12))
ax = plt.axes(projection=proj)

ax.set_extent(
    [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
    crs=ccrs.PlateCarree()
)

ax.coastlines(linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)

gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# -----------------------------
# Initial scatter
# -----------------------------
scat = ax.scatter(
    lon_list[0],
    lat_list[0],
    c=rad_list[0],
    s=SCATTER_SIZE,
    cmap="gray",
    vmin=VMIN,
    vmax=VMAX,
    transform=ccrs.PlateCarree(),
)

cbar = plt.colorbar(
    scat,
    ax=ax,
    label="Radiance",
    fraction=0.02,
    pad=0.01,
    shrink=0.8,
    aspect=40,
)

title = ax.set_title(
    f"AWE Radiance – Frame {frames[0]}"
)

plt.subplots_adjust(
    left=0.04,
    right=0.92,
    bottom=0.05,
    top=0.92,
)

# -----------------------------
# Animation function
# -----------------------------
def update(i):
    lon_v = lon_list[i]
    lat_v = lat_list[i]
    rad_v = rad_list[i]

    if lon_v.size == 0:
        scat.set_offsets(np.empty((0, 2)))
        scat.set_array(np.array([]))
    else:
        scat.set_offsets(np.column_stack((lon_v, lat_v)))
        scat.set_array(rad_v)

    frame_number = frames[i]
    title.set_text(f"AWE Radiance – Frame {frame_number}")

    return scat, title

anim = FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=250,
    blit=False,
)

# -----------------------------
# Show inline with video controls
# -----------------------------
plt.close(fig)
display(HTML(anim.to_html5_video()))

# -----------------------------
# Save as GIF
# -----------------------------
gif_filename = out_dir / "awe_radiance_pacific_centered_every_25_frames.gif"

gif_writer = PillowWriter(fps=FPS)
anim.save(gif_filename, writer=gif_writer)

print(f"Saved GIF to {gif_filename}")

# -----------------------------
# Save as MP4
# -----------------------------
mp4_filename = out_dir / "awe_radiance_pacific_centered_every_25_frames.mp4"

mp4_writer = FFMpegWriter(fps=FPS, bitrate=1800)
anim.save(mp4_filename, writer=mp4_writer)

print(f"Saved MP4 to {mp4_filename}")

ds.close()

Radiance shape: (1756, 256, 256)
Number of frames: 1756
Animating every 25 frames
Number of animation frames: 71


Saved GIF to Animation\awe_radiance_pacific_centered_every_25_frames.gif
Saved MP4 to Animation\awe_radiance_pacific_centered_every_25_frames.mp4


In [18]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML, display
from pathlib import Path

# -----------------------------
# User parameters
# -----------------------------
nc_path = r"Y:\soc\l0c\2024\05\awe_l0c_q20_2024123T0728_02515_v01.nc"

out_dir = Path("Animation")
out_dir.mkdir(parents=True, exist_ok=True)

# Pacific-centered map extent
LON_MIN = 165
LON_MAX = 270
LAT_MIN = -45
LAT_MAX = 55

FRAME_STEP = 25

# Frames to keep fully visible after they appear
KEEP_FRAMES = [0, 475, 1750]

VMIN = 3
VMAX = 20
SCATTER_SIZE = 2.0
FPS = 3

# Moving frame transparency
MOVING_ALPHA = 0.2   # 80% transparent

# Kept frame transparency
KEEP_ALPHA = 1.0

# -----------------------------
# Load dataset
# -----------------------------
ds = Dataset(nc_path, "r")

rad_all = ds.variables["Radiance"][:]
lat_all = ds.variables["Latitude"][:]
lon_raw = ds.variables["Longitude"][:]

# Keep longitude in 0–360 format
lon_all = lon_raw % 360

nt = rad_all.shape[0]
print(f"Radiance shape: {rad_all.shape}")
print(f"Number of frames: {nt}")

frames = list(range(0, nt, FRAME_STEP))
print(f"Animating every {FRAME_STEP} frames")
print(f"Number of animation frames: {len(frames)}")

# -----------------------------
# Helper: extract one frame inside map box
# -----------------------------
def get_frame_data(t):
    lat_t = lat_all[t, :, :]
    lon_t = lon_all[t, :, :]
    rad_t = rad_all[t, :, :]

    lat_flat = lat_t.ravel()
    lon_flat = lon_t.ravel()
    rad_flat = rad_t.ravel()

    valid = (
        np.isfinite(lat_flat)
        & np.isfinite(lon_flat)
        & np.isfinite(rad_flat)
    )

    inside_box = (
        (lat_flat >= LAT_MIN) & (lat_flat <= LAT_MAX)
        & (lon_flat >= LON_MIN) & (lon_flat <= LON_MAX)
    )

    mask = valid & inside_box

    return lon_flat[mask], lat_flat[mask], rad_flat[mask]

# -----------------------------
# Precompute moving-frame data
# -----------------------------
lon_list = []
lat_list = []
rad_list = []

for t in frames:
    lon_v, lat_v, rad_v = get_frame_data(t)
    lon_list.append(lon_v)
    lat_list.append(lat_v)
    rad_list.append(rad_v)

# -----------------------------
# Precompute kept-frame data
# -----------------------------
keep_data = {}

for kf in KEEP_FRAMES:
    if 0 <= kf < nt:
        keep_data[kf] = get_frame_data(kf)
    else:
        print(f"Warning: KEEP_FRAME {kf} is outside available frame range 0–{nt-1}")

# -----------------------------
# Set up figure
# -----------------------------
proj = ccrs.PlateCarree(central_longitude=180)

fig = plt.figure(figsize=(12, 12))
ax = plt.axes(projection=proj)

ax.set_extent(
    [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
    crs=ccrs.PlateCarree()
)

ax.coastlines(linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)

gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# -----------------------------
# Moving transparent scatter
# -----------------------------
scat = ax.scatter(
    lon_list[0],
    lat_list[0],
    c=rad_list[0],
    s=SCATTER_SIZE,
    cmap="gray",
    vmin=VMIN,
    vmax=VMAX,
    alpha=MOVING_ALPHA,
    transform=ccrs.PlateCarree(),
)

# -----------------------------
# Full-opacity kept-frame scatters
# -----------------------------
keep_scatters = {}

for kf in KEEP_FRAMES:
    keep_scatters[kf] = ax.scatter(
        [],
        [],
        c=[],
        s=SCATTER_SIZE,
        cmap="gray",
        vmin=VMIN,
        vmax=VMAX,
        alpha=KEEP_ALPHA,
        transform=ccrs.PlateCarree(),
    )

cbar = plt.colorbar(
    scat,
    ax=ax,
    label="Radiance",
    fraction=0.02,
    pad=0.01,
    shrink=0.8,
    aspect=40,
)

title = ax.set_title(
    f"AWE Radiance – Frame {frames[0]}"
)

plt.subplots_adjust(
    left=0.04,
    right=0.92,
    bottom=0.05,
    top=0.92,
)

# -----------------------------
# Animation function
# -----------------------------
def update(i):
    frame_number = frames[i]

    # Current moving frame: transparent
    lon_v = lon_list[i]
    lat_v = lat_list[i]
    rad_v = rad_list[i]

    if lon_v.size == 0:
        scat.set_offsets(np.empty((0, 2)))
        scat.set_array(np.array([]))
    else:
        scat.set_offsets(np.column_stack((lon_v, lat_v)))
        scat.set_array(rad_v)

    # Kept frames: stay visible after they appear
    active_kept = []

    for kf, keep_scat in keep_scatters.items():
        if frame_number >= kf and kf in keep_data:
            k_lon, k_lat, k_rad = keep_data[kf]

            if k_lon.size == 0:
                keep_scat.set_offsets(np.empty((0, 2)))
                keep_scat.set_array(np.array([]))
            else:
                keep_scat.set_offsets(np.column_stack((k_lon, k_lat)))
                keep_scat.set_array(k_rad)

            active_kept.append(kf)
        else:
            keep_scat.set_offsets(np.empty((0, 2)))
            keep_scat.set_array(np.array([]))

    title.set_text(
        f"AWE Radiance – Frame {frame_number}\n"
        f"Transparent moving swath; kept frames: {active_kept}"
    )

    return [scat, title] + list(keep_scatters.values())

anim = FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=250,
    blit=False,
)

# -----------------------------
# Show inline with video controls
# -----------------------------
plt.close(fig)
display(HTML(anim.to_html5_video()))

# -----------------------------
# Save as GIF
# -----------------------------
gif_filename = out_dir / "awe_radiance_swath_with_kept_frames.gif"

gif_writer = PillowWriter(fps=FPS)
anim.save(gif_filename, writer=gif_writer)

print(f"Saved GIF to {gif_filename}")

# -----------------------------
# Save as MP4
# -----------------------------
mp4_filename = out_dir / "awe_radiance_swath_with_kept_frames.mp4"

mp4_writer = FFMpegWriter(fps=FPS, bitrate=1800)
anim.save(mp4_filename, writer=mp4_writer)

print(f"Saved MP4 to {mp4_filename}")

ds.close()

Radiance shape: (1756, 256, 256)
Number of frames: 1756
Animating every 25 frames
Number of animation frames: 71


Saved GIF to Animation\awe_radiance_swath_with_kept_frames.gif
Saved MP4 to Animation\awe_radiance_swath_with_kept_frames.mp4


## Add 3 subpanels

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML, display
from pathlib import Path

# -----------------------------
# User parameters
# -----------------------------
nc_path = r"Y:\soc\l0c\2024\05\awe_l0c_q20_2024123T0728_02515_v01.nc"

out_dir = Path("Animation")
out_dir.mkdir(parents=True, exist_ok=True)

# Pacific-centered map extent
LON_MIN = 165
LON_MAX = 270
LAT_MIN = -45
LAT_MAX = 55

FRAME_STEP = 50

# Frames to keep on the big map and show below
KEEP_FRAMES = [0, 450, 1750]

# Zoomed subplot size
ZOOM_SIZE_KM = 800.0

VMIN = 3
VMAX = 20
SCATTER_SIZE = 2.0
ZOOM_SCATTER_SIZE = 3.0
FPS = 3

MOVING_ALPHA = 0.2
KEEP_ALPHA = 1.0

# -----------------------------
# Helper functions
# -----------------------------
def get_zoom_extent(center_lat, center_lon, size_km=800):
    half_km = size_km / 2

    dlat = half_km / 111.0
    dlon = half_km / (111.0 * np.cos(np.deg2rad(center_lat)))

    lat_min = center_lat - dlat
    lat_max = center_lat + dlat
    lon_min = center_lon - dlon
    lon_max = center_lon + dlon

    return lon_min, lon_max, lat_min, lat_max


# -----------------------------
# Load dataset
# -----------------------------
ds = Dataset(nc_path, "r")

rad_all = ds.variables["Radiance"][:]
lat_all = ds.variables["Latitude"][:]
lon_raw = ds.variables["Longitude"][:]

center_lat_all = ds.variables["Center_Latitude"][:]
center_lon_raw = ds.variables["Center_Longitude"][:]

# Keep longitude in 0–360 format
lon_all = lon_raw % 360
center_lon_all = center_lon_raw % 360

nt = rad_all.shape[0]
print(f"Radiance shape: {rad_all.shape}")
print(f"Number of frames: {nt}")

frames = list(range(0, nt, FRAME_STEP))
print(f"Animating every {FRAME_STEP} frames")
print(f"Number of animation frames: {len(frames)}")

# -----------------------------
# Helper: extract one frame inside box
# -----------------------------
def get_frame_data(t, lon_min, lon_max, lat_min, lat_max):
    lat_t = lat_all[t, :, :]
    lon_t = lon_all[t, :, :]
    rad_t = rad_all[t, :, :]

    lat_flat = lat_t.ravel()
    lon_flat = lon_t.ravel()
    rad_flat = rad_t.ravel()

    valid = (
        np.isfinite(lat_flat)
        & np.isfinite(lon_flat)
        & np.isfinite(rad_flat)
    )

    inside_box = (
        (lat_flat >= lat_min) & (lat_flat <= lat_max)
        & (lon_flat >= lon_min) & (lon_flat <= lon_max)
    )

    mask = valid & inside_box

    return lon_flat[mask], lat_flat[mask], rad_flat[mask]


# -----------------------------
# Precompute moving-frame data for big map
# -----------------------------
lon_list = []
lat_list = []
rad_list = []

for t in frames:
    lon_v, lat_v, rad_v = get_frame_data(t, LON_MIN, LON_MAX, LAT_MIN, LAT_MAX)
    lon_list.append(lon_v)
    lat_list.append(lat_v)
    rad_list.append(rad_v)

# -----------------------------
# Precompute kept-frame data for big map
# -----------------------------
keep_data = {}

for kf in KEEP_FRAMES:
    if 0 <= kf < nt:
        keep_data[kf] = get_frame_data(kf, LON_MIN, LON_MAX, LAT_MIN, LAT_MAX)
    else:
        print(f"Warning: KEEP_FRAME {kf} is outside available frame range 0–{nt-1}")


# -----------------------------
# Set up figure layout
# -----------------------------
proj = ccrs.PlateCarree(central_longitude=180)

fig = plt.figure(figsize=(12, 15))

gs = fig.add_gridspec(
    nrows=2,
    ncols=3,
    height_ratios=[3.5, 1.2],
    hspace=0.18,
    wspace=0.08,
)

# Big main map
ax_big = fig.add_subplot(gs[0, :], projection=proj)

ax_big.set_extent(
    [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
    crs=ccrs.PlateCarree()
)

ax_big.coastlines(linewidth=0.8)
ax_big.add_feature(cfeature.BORDERS, linewidth=0.5)
ax_big.add_feature(cfeature.STATES, linewidth=0.3)

gl = ax_big.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# Transparent moving scatter
scat = ax_big.scatter(
    lon_list[0],
    lat_list[0],
    c=rad_list[0],
    s=SCATTER_SIZE,
    cmap="gray",
    vmin=VMIN,
    vmax=VMAX,
    alpha=MOVING_ALPHA,
    transform=ccrs.PlateCarree(),
)

# Kept frame scatters on big map
keep_scatters = {}

for kf in KEEP_FRAMES:
    keep_scatters[kf] = ax_big.scatter(
        [],
        [],
        c=[],
        s=SCATTER_SIZE,
        cmap="gray",
        vmin=VMIN,
        vmax=VMAX,
        alpha=KEEP_ALPHA,
        transform=ccrs.PlateCarree(),
    )

cbar = plt.colorbar(
    scat,
    ax=ax_big,
    label="Radiance",
    fraction=0.02,
    pad=0.01,
    shrink=0.8,
    aspect=40,
)

title = ax_big.set_title(f"AWE Radiance – Frame {frames[0]}")

# -----------------------------
# Small zoomed subplots
# -----------------------------
zoom_axes = []

for j, kf in enumerate(KEEP_FRAMES):
    axz = fig.add_subplot(gs[1, j], projection=proj)
    zoom_axes.append(axz)

    if not (0 <= kf < nt):
        axz.set_title(f"Frame {kf} unavailable")
        continue

    center_lat = float(center_lat_all[kf])
    center_lon = float(center_lon_all[kf])

    zlon_min, zlon_max, zlat_min, zlat_max = get_zoom_extent(
        center_lat,
        center_lon,
        size_km=ZOOM_SIZE_KM,
    )

    lon_z, lat_z, rad_z = get_frame_data(
        kf,
        zlon_min,
        zlon_max,
        zlat_min,
        zlat_max,
    )

    axz.set_extent(
        [zlon_min, zlon_max, zlat_min, zlat_max],
        crs=ccrs.PlateCarree()
    )

    axz.coastlines(linewidth=0.6)
    axz.add_feature(cfeature.BORDERS, linewidth=0.4)
    axz.add_feature(cfeature.STATES, linewidth=0.25)

    axz.scatter(
        lon_z,
        lat_z,
        c=rad_z,
        s=ZOOM_SCATTER_SIZE,
        cmap="gray",
        vmin=VMIN,
        vmax=VMAX,
        transform=ccrs.PlateCarree(),
    )

    axz.plot(
        center_lon,
        center_lat,
        "rx",
        markersize=7,
        mew=1.5,
        transform=ccrs.PlateCarree(),
    )

    axz.set_title(
        f"Frame {kf}\nCenter: {center_lat:.1f}°, {center_lon:.1f}°E",
        fontsize=10,
    )

# -----------------------------
# Animation function
# -----------------------------
def update(i):
    frame_number = frames[i]

    # Current moving frame: transparent
    lon_v = lon_list[i]
    lat_v = lat_list[i]
    rad_v = rad_list[i]

    if lon_v.size == 0:
        scat.set_offsets(np.empty((0, 2)))
        scat.set_array(np.array([]))
    else:
        scat.set_offsets(np.column_stack((lon_v, lat_v)))
        scat.set_array(rad_v)

    # Kept frames: stay visible after they appear
    active_kept = []

    for kf, keep_scat in keep_scatters.items():
        if frame_number >= kf and kf in keep_data:
            k_lon, k_lat, k_rad = keep_data[kf]

            if k_lon.size == 0:
                keep_scat.set_offsets(np.empty((0, 2)))
                keep_scat.set_array(np.array([]))
            else:
                keep_scat.set_offsets(np.column_stack((k_lon, k_lat)))
                keep_scat.set_array(k_rad)

            active_kept.append(kf)
        else:
            keep_scat.set_offsets(np.empty((0, 2)))
            keep_scat.set_array(np.array([]))

    title.set_text(
        f"AWE Radiance – Frame {frame_number}\n"
        f"Transparent moving swath; kept frames: {active_kept}"
    )

    return [scat, title] + list(keep_scatters.values())


anim = FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=250,
    blit=False,
)

# -----------------------------
# Show inline with video controls
# -----------------------------
plt.close(fig)
display(HTML(anim.to_html5_video()))

# -----------------------------
# Save as GIF
# -----------------------------
gif_filename = out_dir / "awe_radiance_swath_with_zoom_panels.gif"

gif_writer = PillowWriter(fps=FPS)
anim.save(gif_filename, writer=gif_writer)

print(f"Saved GIF to {gif_filename}")

# -----------------------------
# Save as MP4
# -----------------------------
mp4_filename = out_dir / "awe_radiance_swath_with_zoom_panels.mp4"

mp4_writer = FFMpegWriter(fps=FPS, bitrate=1800)
anim.save(mp4_filename, writer=mp4_writer)

print(f"Saved MP4 to {mp4_filename}")

ds.close()

Radiance shape: (1756, 256, 256)
Number of frames: 1756
Animating every 50 frames
Number of animation frames: 36


C:\Users\domin\.conda\envs\juwavelet_env\Lib\site-packages\cartopy\io\__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces_lakes.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


Saved GIF to Animation\awe_radiance_swath_with_zoom_panels.gif
Saved MP4 to Animation\awe_radiance_swath_with_zoom_panels.mp4


## Coloring

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation, FFMpegWriter
from pathlib import Path
import warnings
from matplotlib.lines import Line2D

warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in create_collection",
    category=RuntimeWarning,
)

# -----------------------------
# User parameters
# -----------------------------
# nc_path = r"Y:\soc\l0c\2024\05\awe_l0c_q20_2024123T0728_02515_v01.nc"
nc_path = r"/Users/anhphan/Downloads/awe_l0c_q20_2024123T0728_02515_v01.nc"

out_dir = Path("Animation")
out_dir.mkdir(parents=True, exist_ok=True)

LON_MIN = 165
LON_MAX = 270
LAT_MIN = -45
LAT_MAX = 55

FRAME_STEP = 10
PIXEL_SKIP = 2

KEEP_FRAMES = [0, 500, 1750]
ZOOM_SIZE_KM = 800.0

VMIN = 3
VMAX = 22

TRAIL_SCATTER_SIZE = 0.4
CURRENT_SCATTER_SIZE = 1.5
KEEP_SCATTER_SIZE = 2.5
ZOOM_SCATTER_SIZE = 3.0

FPS = 10

TRAIL_ALPHA = 0.35
COLORED_TRAIL_ALPHA = 0.60
CURRENT_ALPHA = 1.0
KEEP_ALPHA = 1.0

SOLAR_PANEL_RANGES = [(0, 100), (1550, 10**9)]

ANIMATION_TRAIL_COLOR = "lightgray"
NORMAL_TRAIL_COLOR = "lightblue"
SOLAR_PANEL_TRAIL_COLOR = "lightcoral"

LABEL_LON_OFFSET = -6.0
LABEL_LAT_OFFSET = 5.0


# -----------------------------
# Helper functions
# -----------------------------
def get_zoom_extent(center_lat, center_lon, size_km=800):
    half_km = size_km / 2.0
    dlat = half_km / 111.0
    dlon = half_km / (111.0 * np.cos(np.deg2rad(center_lat)))

    return (
        center_lon - dlon,
        center_lon + dlon,
        center_lat - dlat,
        center_lat + dlat,
    )


def is_solar_panel_frame(frame):
    return any(start <= frame <= end for start, end in SOLAR_PANEL_RANGES)


def get_frame_data(t, lon_min, lon_max, lat_min, lat_max):
    lat_t = lat_all[t, ::PIXEL_SKIP, ::PIXEL_SKIP]
    lon_t = lon_all[t, ::PIXEL_SKIP, ::PIXEL_SKIP]
    rad_t = rad_all[t, ::PIXEL_SKIP, ::PIXEL_SKIP]

    lat_flat = lat_t.ravel()
    lon_flat = lon_t.ravel()
    rad_flat = rad_t.ravel()

    valid = (
        np.isfinite(lat_flat)
        & np.isfinite(lon_flat)
        & np.isfinite(rad_flat)
    )

    inside_box = (
        (lat_flat >= lat_min) & (lat_flat <= lat_max)
        & (lon_flat >= lon_min) & (lon_flat <= lon_max)
    )

    mask = valid & inside_box

    return lon_flat[mask], lat_flat[mask], rad_flat[mask]


def add_map_features(ax, linewidth_scale=1.0):
    ax.coastlines(linewidth=0.8 * linewidth_scale)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5 * linewidth_scale)
    ax.add_feature(cfeature.STATES, linewidth=0.3 * linewidth_scale)


def set_keep_frames_visible(visible):
    for kf in KEEP_FRAMES:
        if kf in keep_scatters:
            keep_scatters[kf].set_visible(visible)
        if kf in keep_labels:
            keep_labels[kf].set_visible(visible)


def set_trail_gray():
    for scat in trail_scatters:
        scat.set_color(ANIMATION_TRAIL_COLOR)
        scat.set_alpha(TRAIL_ALPHA)


def set_trail_colored():
    for j, scat in enumerate(trail_scatters):
        frame_number = frames[j]

        if is_solar_panel_frame(frame_number):
            scat.set_color(SOLAR_PANEL_TRAIL_COLOR)
            scat.set_alpha(COLORED_TRAIL_ALPHA)
        else:
            scat.set_color(NORMAL_TRAIL_COLOR)
            scat.set_alpha(TRAIL_ALPHA)


# -----------------------------
# Load dataset
# -----------------------------
ds = Dataset(nc_path, "r")

rad_all = ds.variables["Radiance"][:]
lat_all = ds.variables["Latitude"][:]
lon_raw = ds.variables["Longitude"][:]

center_lat_all = ds.variables["Center_Latitude"][:]
center_lon_raw = ds.variables["Center_Longitude"][:]

lon_all = lon_raw % 360
center_lon_all = center_lon_raw % 360

nt = rad_all.shape[0]

SOLAR_PANEL_RANGES = [(0, 100), (1550, nt - 1)]

print(f"Radiance shape: {rad_all.shape}")
print(f"Number of frames: {nt}")

frames = list(range(0, nt, FRAME_STEP))

print(f"Animating every {FRAME_STEP} frames")
print(f"Pixel skip: {PIXEL_SKIP}")
print(f"Number of animation frames: {len(frames)}")
print(f"Solar-panel frame ranges: {SOLAR_PANEL_RANGES}")

# -----------------------------
# Projections
# -----------------------------
main_proj = ccrs.LambertAzimuthalEqualArea(
    central_longitude=220,
    central_latitude=5,
)

data_crs = ccrs.PlateCarree()

# -----------------------------
# Precompute main-map frame data
# -----------------------------
lon_list = []
lat_list = []
rad_list = []

for t in frames:
    lon_v, lat_v, rad_v = get_frame_data(
        t,
        LON_MIN,
        LON_MAX,
        LAT_MIN,
        LAT_MAX,
    )
    lon_list.append(lon_v)
    lat_list.append(lat_v)
    rad_list.append(rad_v)

# -----------------------------
# Precompute kept-frame data
# -----------------------------
keep_data = {}

for kf in KEEP_FRAMES:
    if 0 <= kf < nt:
        keep_data[kf] = get_frame_data(
            kf,
            LON_MIN,
            LON_MAX,
            LAT_MIN,
            LAT_MAX,
        )
    else:
        print(f"Warning: KEEP_FRAME {kf} is outside available frame range 0–{nt-1}")

# -----------------------------
# Figure layout
# -----------------------------
fig = plt.figure(figsize=(12, 9), dpi=100)

gs = fig.add_gridspec(
    nrows=3,
    ncols=2,
    width_ratios=[4.2, 1.35],
    height_ratios=[1, 1, 1],
    hspace=0.25,
    wspace=0.10,
)

# -----------------------------
# Big main map
# -----------------------------
ax_big = fig.add_subplot(gs[:, 0], projection=main_proj)

ax_big.set_extent(
    [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
    crs=data_crs,
)

add_map_features(ax_big)

gl = ax_big.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# One gray trail scatter per animation frame
trail_scatters = []

for j, frame_number in enumerate(frames):
    scat = ax_big.scatter(
        [],
        [],
        color=ANIMATION_TRAIL_COLOR,
        s=TRAIL_SCATTER_SIZE,
        alpha=TRAIL_ALPHA,
        transform=data_crs,
        zorder=1 + j * 0.001,
    )
    trail_scatters.append(scat)

# Current frame
current_scat = ax_big.scatter(
    [],
    [],
    c=[],
    s=CURRENT_SCATTER_SIZE,
    cmap="gray",
    vmin=VMIN,
    vmax=VMAX,
    alpha=CURRENT_ALPHA,
    transform=data_crs,
    zorder=10,
)

# Selected frames
keep_scatters = {}
keep_labels = {}

for kf in KEEP_FRAMES:
    keep_scatters[kf] = ax_big.scatter(
        [],
        [],
        c=[],
        s=KEEP_SCATTER_SIZE,
        cmap="gray",
        vmin=VMIN,
        vmax=VMAX,
        alpha=KEEP_ALPHA,
        transform=data_crs,
        zorder=20,
    )

    if 0 <= kf < nt:
        label_lon = float(center_lon_all[kf]) + LABEL_LON_OFFSET
        label_lat = float(center_lat_all[kf]) + LABEL_LAT_OFFSET

        keep_labels[kf] = ax_big.text(
            label_lon,
            label_lat,
            f"Frame {kf}",
            fontsize=9,
            fontweight="bold",
            ha="center",
            va="bottom",
            transform=data_crs,
            bbox=dict(
                facecolor="white",
                edgecolor="black",
                alpha=0.8,
                boxstyle="round,pad=0.25",
            ),
            zorder=30,
        )
        keep_labels[kf].set_visible(False)

cbar = plt.colorbar(
    current_scat,
    ax=ax_big,
    label="Radiance",
    fraction=0.015,
    pad=0.01,
    shrink=0.8,
    aspect=35,
)

title = ax_big.set_title("AWE Radiance")

# -----------------------------
# Small zoomed subplots on right
# -----------------------------
for j, kf in enumerate(KEEP_FRAMES):
    if not (0 <= kf < nt):
        axz = fig.add_subplot(gs[j, 1], projection=main_proj)
        axz.set_title(f"Frame {kf} unavailable")
        continue

    center_lat = float(center_lat_all[kf])
    center_lon = float(center_lon_all[kf])

    zoom_proj = ccrs.LambertAzimuthalEqualArea(
        central_longitude=center_lon,
        central_latitude=center_lat,
    )

    axz = fig.add_subplot(gs[j, 1], projection=zoom_proj)

    zlon_min, zlon_max, zlat_min, zlat_max = get_zoom_extent(
        center_lat,
        center_lon,
        size_km=ZOOM_SIZE_KM,
    )

    lon_z, lat_z, rad_z = get_frame_data(
        kf,
        zlon_min,
        zlon_max,
        zlat_min,
        zlat_max,
    )

    axz.set_extent(
        [zlon_min, zlon_max, zlat_min, zlat_max],
        crs=data_crs,
    )

    add_map_features(axz, linewidth_scale=0.7)

    axz.scatter(
        lon_z,
        lat_z,
        c=rad_z,
        s=ZOOM_SCATTER_SIZE,
        cmap="gray",
        vmin=VMIN,
        vmax=VMAX,
        transform=data_crs,
        zorder=3,
    )

    axz.text(
        0.5,
        1.04,
        f"Frame {kf}",
        transform=axz.transAxes,
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="black",
            alpha=0.8,
            boxstyle="round,pad=0.25",
        ),
    )


# -----------------------------
# Animation function
# -----------------------------
def update(i):
    frame_number = frames[i]

    # Trail stays gray in the animation
    for j, scat in enumerate(trail_scatters):
        if j < i and frames[j] not in KEEP_FRAMES:
            lon_j = lon_list[j]
            lat_j = lat_list[j]

            if lon_j.size == 0:
                scat.set_offsets(np.empty((0, 2)))
            else:
                scat.set_offsets(np.column_stack((lon_j, lat_j)))
        else:
            scat.set_offsets(np.empty((0, 2)))

    # Current frame
    current_lon = lon_list[i]
    current_lat = lat_list[i]
    current_rad = rad_list[i]

    if current_lon.size == 0:
        current_scat.set_offsets(np.empty((0, 2)))
        current_scat.set_array(np.array([]))
    else:
        current_scat.set_offsets(np.column_stack((current_lon, current_lat)))
        current_scat.set_array(current_rad)

    # Selected frames stay visible after they appear
    for kf, keep_scat in keep_scatters.items():
        if frame_number >= kf and kf in keep_data:
            k_lon, k_lat, k_rad = keep_data[kf]

            if k_lon.size == 0:
                keep_scat.set_offsets(np.empty((0, 2)))
                keep_scat.set_array(np.array([]))
            else:
                keep_scat.set_offsets(np.column_stack((k_lon, k_lat)))
                keep_scat.set_array(k_rad)

            if kf in keep_labels:
                keep_labels[kf].set_visible(True)
        else:
            keep_scat.set_offsets(np.empty((0, 2)))
            keep_scat.set_array(np.array([]))

            if kf in keep_labels:
                keep_labels[kf].set_visible(False)

    title.set_text("AWE Radiance")

    return (
        trail_scatters
        + [current_scat, title]
        + list(keep_scatters.values())
        + list(keep_labels.values())
    )


anim = FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=250,
    blit=False,
)

# -----------------------------
# Save MP4
# -----------------------------
mp4_filename = out_dir / "awe_radiance_gray_trail_subplots_right.mp4"

mp4_writer = FFMpegWriter(fps=FPS)
anim.save(mp4_filename, writer=mp4_writer)

print(f"Saved MP4 to {mp4_filename}")

# -----------------------------
# Save final frame PNG version 1:
# Gray trail, keep frames visible
# -----------------------------
last_i = len(frames) - 1

update(last_i)
set_trail_gray()
set_keep_frames_visible(True)
current_scat.set_visible(True)
title.set_text("AWE Radiance")

last_frame_gray_png = out_dir / "awe_radiance_final_gray_trail_with_keep_frames.png"
fig.savefig(last_frame_gray_png, dpi=300, bbox_inches="tight")

print(f"Saved gray final frame PNG to {last_frame_gray_png}")

# -----------------------------
# Save final frame PNG version 2:
# Colored trail, grayscale keep/current overlays removed
#
# Important:
# - KEEP_FRAME overlays are hidden.
# - current_scat is hidden, so the final frame does not stay grayscale.
# - The trail scatters are kept, so frame 1750 becomes blue/red according to
#   the trail coloring.
# -----------------------------
# -----------------------------
# Save final frame PNG version 2:
# Colored trail, grayscale keep/current overlays removed
# -----------------------------
update(last_i)
set_keep_frames_visible(False)
set_trail_colored()
current_scat.set_visible(False)
title.set_text("AWE Radiance")

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markersize=8,
        markerfacecolor=NORMAL_TRAIL_COLOR,
        markeredgecolor="none",
        label="No solar-panel frames",
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markersize=8,
        markerfacecolor=SOLAR_PANEL_TRAIL_COLOR,
        markeredgecolor="none",
        label="Solar-panel frames",
    ),
]

legend = ax_big.legend(
    handles=legend_handles,
    loc="lower right",
    fontsize=11,
    frameon=True,
    framealpha=0.95,
    borderpad=0.8,
)

legend.set_zorder(100)

last_frame_colored_png = out_dir / "awe_radiance_final_colored_trail_no_keep_frames.png"
fig.savefig(last_frame_colored_png, dpi=300, bbox_inches="tight")

print(f"Saved colored final frame PNG to {last_frame_colored_png}")

plt.close(fig)
ds.close()

Radiance shape: (1756, 256, 256)
Number of frames: 1756
Animating every 10 frames
Pixel skip: 2
Number of animation frames: 176
Solar-panel frame ranges: [(0, 100), (1550, 1755)]
Saved MP4 to Animation/awe_radiance_gray_trail_subplots_right.mp4
Saved gray final frame PNG to Animation/awe_radiance_final_gray_trail_with_keep_frames.png
Saved colored final frame PNG to Animation/awe_radiance_final_colored_trail_no_keep_frames.png


## MLCloud Plot

In [23]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature


# -----------------------------
# User parameters
# -----------------------------
nc_path = r"/Users/anhphan/Downloads/awe_l3c_q20_2024123T0730_02515_v01.nc"

out_dir = Path("MLCloud_Plots")
out_dir.mkdir(parents=True, exist_ok=True)

radiance_png = out_dir / "awe_l3c_02515_radiance_map_crop_last15.png"
mlcloud_png = out_dir / "awe_l3c_02515_mlcloud_map_crop_last15.png"

PIXEL_SKIP = 2
SCATTER_SIZE = 1.0

RAD_VMIN = 3
RAD_VMAX = 22

MLCLOUD_VMIN = 0
MLCLOUD_VMAX = 100

LON_MIN = 165
LON_MAX = 270
LAT_MIN = -45
LAT_MAX = 55

REMOVE_LAST_FRACTION = 0.15
CROP_DIRECTION = "x"


# -----------------------------
# Helper functions
# -----------------------------
def squeeze_to_2d(arr, name):
    arr = np.asarray(arr, dtype=float)

    if arr.ndim == 3:
        if arr.shape[-1] == 1:
            arr = arr[:, :, 0]
        elif arr.shape[0] == 1:
            arr = arr[0, :, :]
        else:
            raise ValueError(f"Unexpected {name} shape: {arr.shape}")

    if arr.ndim != 2:
        raise ValueError(f"Unexpected {name} shape after squeeze: {arr.shape}")

    return arr


def crop_last_fraction(data, lat, lon, remove_fraction=0.15, direction="x"):
    if direction.lower() == "x":
        nx = data.shape[1]
        keep_nx = int(nx * (1.0 - remove_fraction))

        data = data[:, :keep_nx]
        lat = lat[:, :keep_nx]
        lon = lon[:, :keep_nx]

        print(f"Removed last {remove_fraction * 100:.0f}% in x direction")
        print(f"Keeping first {keep_nx} of {nx} columns")

    elif direction.lower() == "y":
        ny = data.shape[0]
        keep_ny = int(ny * (1.0 - remove_fraction))

        data = data[:keep_ny, :]
        lat = lat[:keep_ny, :]
        lon = lon[:keep_ny, :]

        print(f"Removed last {remove_fraction * 100:.0f}% in y direction")
        print(f"Keeping first {keep_ny} of {ny} rows")

    else:
        raise ValueError("CROP_DIRECTION must be either 'x' or 'y'")

    return data, lat, lon


def get_map_data(data, lat, lon):
    data_flat = data[::PIXEL_SKIP, ::PIXEL_SKIP].ravel()
    lat_flat = lat[::PIXEL_SKIP, ::PIXEL_SKIP].ravel()
    lon_flat = lon[::PIXEL_SKIP, ::PIXEL_SKIP].ravel()

    valid = (
        np.isfinite(data_flat)
        & np.isfinite(lat_flat)
        & np.isfinite(lon_flat)
    )

    inside_box = (
        (lon_flat >= LON_MIN)
        & (lon_flat <= LON_MAX)
        & (lat_flat >= LAT_MIN)
        & (lat_flat <= LAT_MAX)
    )

    mask = valid & inside_box

    return lon_flat[mask], lat_flat[mask], data_flat[mask]


def add_map_features(ax):
    ax.coastlines(linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.STATES, linewidth=0.3)

    gl = ax.gridlines(
        draw_labels=True,
        linestyle="--",
        alpha=0.5,
    )

    gl.top_labels = False
    gl.right_labels = False


def save_map(
    lon_plot,
    lat_plot,
    data_plot,
    title,
    colorbar_label,
    cmap,
    vmin,
    vmax,
    output_path,
):
    fig = plt.figure(figsize=(10, 8), dpi=150)
    ax = fig.add_subplot(1, 1, 1, projection=main_proj)

    ax.set_extent(
        [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
        crs=data_crs,
    )

    add_map_features(ax)

    sc = ax.scatter(
        lon_plot,
        lat_plot,
        c=data_plot,
        s=SCATTER_SIZE,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        transform=data_crs,
    )

    plt.colorbar(
        sc,
        ax=ax,
        label=colorbar_label,
        fraction=0.015,
        pad=0.01,
        shrink=0.8,
        aspect=35,
    )

    ax.set_title(title)

    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {output_path}")

    plt.close(fig)


# -----------------------------
# Load dataset
# -----------------------------
with Dataset(nc_path, "r") as ds:
    radiance = squeeze_to_2d(ds.variables["Radiance"][:], "Radiance")
    mlcloud = squeeze_to_2d(ds.variables["MLCloud"][:], "MLCloud")
    lat = squeeze_to_2d(ds.variables["Latitude"][:], "Latitude")
    lon = squeeze_to_2d(ds.variables["Longitude"][:], "Longitude")

print(f"Original Radiance shape: {radiance.shape}")
print(f"Original MLCloud shape: {mlcloud.shape}")
print(f"Original Latitude shape: {lat.shape}")
print(f"Original Longitude shape: {lon.shape}")

lon = lon % 360

# -----------------------------
# Crop last 15% of swath
# -----------------------------
radiance, lat_crop, lon_crop = crop_last_fraction(
    radiance,
    lat,
    lon,
    remove_fraction=REMOVE_LAST_FRACTION,
    direction=CROP_DIRECTION,
)

mlcloud, _, _ = crop_last_fraction(
    mlcloud,
    lat,
    lon,
    remove_fraction=REMOVE_LAST_FRACTION,
    direction=CROP_DIRECTION,
)

print(f"Cropped Radiance shape: {radiance.shape}")
print(f"Cropped MLCloud shape: {mlcloud.shape}")
print(f"Cropped Latitude shape: {lat_crop.shape}")
print(f"Cropped Longitude shape: {lon_crop.shape}")

# -----------------------------
# Prepare map data
# -----------------------------
rad_lon, rad_lat, rad_plot = get_map_data(radiance, lat_crop, lon_crop)
ml_lon, ml_lat, ml_plot = get_map_data(mlcloud, lat_crop, lon_crop)

print(f"Radiance valid points plotted: {rad_plot.size}")
print(f"MLCloud valid points plotted: {ml_plot.size}")

# -----------------------------
# Projection: same as radiance script
# -----------------------------
main_proj = ccrs.LambertAzimuthalEqualArea(
    central_longitude=220,
    central_latitude=5,
)

data_crs = ccrs.PlateCarree()

# -----------------------------
# Save separate images
# -----------------------------
save_map(
    rad_lon,
    rad_lat,
    rad_plot,
    title="AWE Radiance",
    colorbar_label="Radiance",
    cmap="gray",
    vmin=RAD_VMIN,
    vmax=RAD_VMAX,
    output_path=radiance_png,
)

save_map(
    ml_lon,
    ml_lat,
    ml_plot,
    title="AWE MLCloud",
    colorbar_label="MLCloud",
    cmap="viridis",
    vmin=MLCLOUD_VMIN,
    vmax=MLCLOUD_VMAX,
    output_path=mlcloud_png,
)

print("Done.")

Original Radiance shape: (451, 6751)
Original MLCloud shape: (451, 6751)
Original Latitude shape: (451, 6751)
Original Longitude shape: (451, 6751)
Removed last 15% in x direction
Keeping first 5738 of 6751 columns
Removed last 15% in x direction
Keeping first 5738 of 6751 columns
Cropped Radiance shape: (451, 5738)
Cropped MLCloud shape: (451, 5738)
Cropped Latitude shape: (451, 5738)
Cropped Longitude shape: (451, 5738)
Radiance valid points plotted: 412083
MLCloud valid points plotted: 412083
Saved: MLCloud_Plots/awe_l3c_02515_radiance_map_crop_last15.png
Saved: MLCloud_Plots/awe_l3c_02515_mlcloud_map_crop_last15.png
Done.


## Training Dataset Plots

In [16]:

import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# Settings
# =========================
OUT_DIR = Path("training_data_pie_plots")
OUT_DIR.mkdir(exist_ok=True)

DPI = 300
FIGSIZE = (5, 4)

COLORS_2CLASS = ["#4C78A8", "#F58518"]
COLORS_SPLIT = ["#54A24B", "#E45756", "#72B7B2"]


# =========================
# Helper function
# =========================
def make_pie_plot(
    labels,
    values,
    title,
    outfile,
    colors,
    autopct_func=None,
):
    total = sum(values)

    if autopct_func is None:
        autopct_func = lambda pct: (
            f"{pct:.1f}%\n({int(round(pct * total / 100)):,})"
        )

    fig, ax = plt.subplots(figsize=FIGSIZE)

    wedges, texts, autotexts = ax.pie(
        values,
        labels=labels,
        colors=colors,
        autopct=autopct_func,
        startangle=90,
        counterclock=False,
        textprops={"fontsize": 11},
        wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    )

    for t in autotexts:
        t.set_fontsize(10)
        t.set_color("white")
        t.set_weight("bold")

    ax.set_title(title, fontsize=15, weight="bold")
    ax.axis("equal")

    plt.tight_layout()
    fig.savefig(OUT_DIR / outfile, dpi=DPI, bbox_inches="tight")
    fig.savefig(OUT_DIR / outfile.replace(".png", ".pdf"), bbox_inches="tight")
    plt.close(fig)


# =========================
# 1. Cloud Dataset
# =========================
make_pie_plot(
    labels=["No Cloud\nClass 0", "Cloud\nClass 1"],
    values=[32822, 32780],
    title="Cloud Dataset",
    outfile="cloud_dataset_pie.png",
    colors=COLORS_2CLASS,
)


# =========================
# 2. Solar Panel Dataset
# =========================
make_pie_plot(
    labels=["No Solar Panel", "Solar Panel"],
    values=[114938, 61085],
    title="Solar Panel Dataset",
    outfile="solar_panel_dataset_pie.png",
    colors=COLORS_2CLASS,
)


# =========================
# 3. Data Split
# =========================
make_pie_plot(
    labels=["Training", "Testing", "Validation"],
    values=[70, 20, 10],
    title="Data Split",
    outfile="data_split_pie.png",
    colors=COLORS_SPLIT,
    autopct_func=lambda pct: f"{pct:.0f}%",
)


print(f"Saved plots to: {OUT_DIR.resolve()}")


Saved plots to: /Users/anhphan/juwavelet_awe/AWE Phase Speed Notebooks/training_data_pie_plots
